# 07 — Verification: stress-testing the three surprising results

**Question this notebook answers:** are the three counter-intuitive findings of
this project real, or artefacts of how they were measured?

Notebooks 01–06 produced three results that contradict the expectations the
project was built on. Each is worth an experiment before it goes in a README,
because each has a plausible boring explanation:

| Finding | The boring explanation to rule out |
|---|---|
| RemoteCLIP zero-shot (0.187 F1) is far **worse** than generic CLIP (0.335) | we transferred CLIP's best prompt strategy to RemoteCLIP instead of tuning per model |
| Natural-language class names **hurt** (v2 → v3 = −0.062) | wording and ensembling were confounded; the 2×2 design was never completed |
| Masking the atmospheric bands (B01/B09/B10) costs **more** F1 than masking NIR | the model is keying on per-scene acquisition conditions, i.e. the random split leaks |

**Experiments here**

- **A** — full prompt sweep on *both* models, completing the wording × ensembling
  2×2 factorial.
- **B** — the text tower vs the image tower: does RemoteCLIP fail because its
  *features* are bad, or because its *text prototypes* are misplaced?
- **C** — leakage quantified: a spatially-blocked split proxy built from
  atmospheric bands, and the accuracy drop it causes.
- **D** *(optional, adds ~25 min)* — the two NB02 ablations at 3 seeds, so the
  stem and multispectral deltas get real error bars.

**Expected runtime:** ~25 minutes for A–C, ~50 with D.

**A note on the test set.** These are *confirmatory* analyses of results already
reported, not a new search for a better model. No architecture, hyperparameter
or checkpoint is selected here on the basis of test performance. The one
exception is inherited and stated in NB03: the best prompt strategy was chosen
on test, which is why section A reports every strategy rather than only a winner.

### Environment setup and persistence

On Colab this clones the repository, installs the pinned dependencies, and — the
part that matters — mounts Google Drive so that **outputs survive the session**.
Locally it is a no-op beyond putting `src/` on the path.

**Why Drive.** A Colab VM is deleted when the session ends, and the notebooks
depend on each other's artefacts: 01 writes the split and normalisation
statistics, 02 writes the model checkpoint that 04, 05 and 06 all load. Without
persistence, a disconnect means re-running earlier chapters. So `outputs/` and
`figures/` are redirected to Drive via environment variables read by
`s2map.config` at import time.

**`data/` is deliberately NOT on Drive.** The EuroSAT cache is ~2.9 GB and every
training epoch reads it randomly; Drive is a network mount and random reads
through it are slow enough to dominate the runtime. Re-downloading into the
local VM disk each session costs a few minutes and is the faster trade. Set
`USE_DRIVE = False` to keep everything ephemeral.

The install is wrapped so a fragile wheel prints a clear message instead of
killing the kernel halfway through a 40-minute session.

**Re-running this cell picks up code changes.** It hard-resets the clone to
`origin/main` and purges `s2map` from `sys.modules`, because Python caches
imported modules: without the purge, a `git pull` updates the file on disk while
the kernel keeps running the old code, and you get an `AttributeError` for a
function that is visibly there in the source. The clone is treated as a
read-only checkout — edit the notebook in Colab or the code locally, not in
`/content/s2-chips-to-map`, since the reset discards changes there.

**If this cell reports a numpy mismatch**, restart the runtime
(*Runtime → Restart session*) and run it again. Colab preinstalls a mutually
consistent numpy/torch/scipy set; when pip replaces one of them on disk, the
kernel is left holding the old version in memory and every compiled extension
raises `ValueError: numpy.dtype size changed`. Only a restart fixes it. The
requirements file uses compatible *ranges* rather than exact pins for exactly
this group of packages, so it should not happen — the check is there because it
is the single most common way a Colab notebook dies.

In [ ]:
# --- edit these two if you are running your own fork -----------------------
GITHUB_USER = "SaadH-077"
USE_DRIVE = True          # False -> everything stays in the ephemeral session
# ---------------------------------------------------------------------------

import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "s2-chips-to-map"

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PERSIST = Path("/content/drive/MyDrive/s2-chips-to-map")
        for sub in ("outputs", "figures"):
            (PERSIST / sub).mkdir(parents=True, exist_ok=True)
        # Read by s2map.config at import time, so this must happen before the import below.
        os.environ["S2MAP_OUTPUT_DIR"] = str(PERSIST / "outputs")
        os.environ["S2MAP_FIGURE_DIR"] = str(PERSIST / "figures")
        print("persisting outputs and figures to", PERSIST)

    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=False)
    if Path(REPO).exists():
        os.chdir(REPO)
        # Pick up fixes without needing a fresh VM: a stale clone from an earlier
        # session is otherwise invisible and confusing. --depth 1 clones are
        # shallow, so unshallow first or the pull fails on a diverged history.
        subprocess.run(["git", "fetch", "--quiet", "--depth", "50", "origin", "main"], check=False)
        before = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        subprocess.run(["git", "reset", "--hard", "--quiet", "origin/main"], check=False)
        after = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        if before != after:
            print(f"repo updated {before[:7]} -> {after[:7]}")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    except subprocess.CalledProcessError as exc:
        print("!! dependency install failed:", exc)
        print("!! continuing anyway — the cells below will report what is missing")

ROOT = Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Drop any already-imported s2map modules before importing them again. Python
# caches modules in sys.modules, so a `git pull` that updates a file on disk has
# NO effect on a kernel that already imported it — you get
# `AttributeError: module 's2map.bands' has no attribute ...` for a function you
# can plainly see in the source. Purging here means re-running this one cell is
# enough to pick up new code, without restarting the runtime and losing state.
_stale = [m for m in list(sys.modules) if m == "s2map" or m.startswith("s2map.")]
for _mod in _stale:
    del sys.modules[_mod]
if _stale:
    print(f"reloaded {len(_stale)} s2map modules from disk (was: {', '.join(sorted(_stale))})")

from s2map import config as C
C.ensure_dirs()
C.print_environment()
print()
# Catches the one Colab failure mode that no amount of re-running fixes:
# pip replaced numpy on disk after this kernel already imported it.
C.check_environment()
cfg = C.load_config()
SEED = C.set_seed(cfg.seed)
print(f"seed             {SEED}   (multi-seed runs use {cfg.seeds})")
DEVICE = C.get_device()
print(f"device           {DEVICE}")
print(f"outputs ->       {C.OUTPUT_DIR}")
print(f"figures ->       {C.FIGURE_DIR}")
print(f"data cache ->    {C.DATA_DIR}  (local/ephemeral by design — see the note above)")
if DEVICE == "cpu":
    print("\n!! no GPU detected. On Colab: Runtime > Change runtime type > T4 GPU.")

### Load everything the earlier chapters produced

Reuses NB04's cached frozen features where available, so nothing is re-encoded
unnecessarily. Both CLIP models still have to be loaded because sections A and B
need their **text** encoders, which were never cached.

In [ ]:
import time
import numpy as np
import torch
from s2map import bands as B, clip_utils as CL, data as D, evaluate as E, models as M, viz as V

V.set_style()
bundle = D.load_eurosat_ms()
X, y = bundle.X, bundle.y
splits = D.load_splits()
stats = D.load_band_stats()
test_idx, train_idx = splits["test"], splits["train"]
y_test = y[test_idx]

FEATURE_DIR = C.OUTPUT_DIR / "features"
prior = E.load_results()
def prior_value(nb, arm, key="test_macro_f1"):
    for r in prior:
        if r.get("notebook") == nb and r.get("arm") == arm:
            v = r.get(key)
            return v["mean"] if isinstance(v, dict) else v
    return None

print("prior results to verify:")
print(f"  CLIP zero-shot (v4)      {prior_value('03', 'clip_vitb32_v4_prompt_ensemble'):.4f}")
print(f"  RemoteCLIP zero-shot     {prior_value('03', 'remoteclip_vitb32'):.4f}")
print(f"  supervised ceiling       {prior_value('02', '_ceiling'):.4f}")
print(f"  test chips               {test_idx.size:,}")

Both CLIP variants are loaded for their **text** encoders. If RemoteCLIP fails to
load here the notebook still runs, but sections A and B lose their comparison arm
and the verification is inconclusive rather than negative — the cells say so.

In [ ]:
models = {}
models["CLIP"] = CL.load_openclip("ViT-B-32", "openai", device=DEVICE)
try:
    rs_model, rs_prep, rs_tok, info = CL.load_remoteclip("ViT-B-32", device=DEVICE)
    models["RemoteCLIP"] = (rs_model, rs_prep, rs_tok)
    print(f"RemoteCLIP loaded, missing keys: {len(info['missing'])}")
except Exception as exc:
    print("!! RemoteCLIP unavailable:", type(exc).__name__, exc)

CACHE = {"CLIP": "clip_vitb32", "RemoteCLIP": "remoteclip_vitb32"}

def image_features(name):
    """All 27,000 image features for a model, from NB04's cache if present."""
    path = FEATURE_DIR / f"{CACHE[name]}.npz"
    if path.exists():
        with np.load(path) as f:
            feats = f["features"]
        print(f"{name:<12} features from cache {feats.shape}")
        return feats
    model, preprocess, _ = models[name]
    out, t0 = [], time.time()
    for s in range(0, y.size, 1024):
        block = np.asarray(X[s:s + 1024])
        images = CL.preprocess_chips(block, preprocess, batch=256)
        out.append(CL.encode_images(model, images, device=DEVICE, batch=256).numpy())
    feats = np.concatenate(out)
    FEATURE_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, features=feats.astype(np.float32))
    print(f"{name:<12} encoded {feats.shape} in {time.time() - t0:.0f}s")
    return feats

FEATS = {name: image_features(name) for name in models}
for name, f in FEATS.items():
    assert f.shape[0] == y.size, (name, f.shape)

## A — The prompt sweep, on both models, as a proper 2×2

NB03 evaluated four strategies on CLIP and then applied only the winner to
RemoteCLIP. Two things were left undone, and both matter:

1. **RemoteCLIP never got its own prompt sweep.** If it simply prefers different
   wording, its apparent collapse is a measurement artefact rather than a
   property of the model.
2. **The wording × ensembling design was incomplete.** We had raw+single (v2),
   natural+single (v3) and natural+ensemble (v4) — but never raw+ensemble. Without
   that fourth cell you cannot separate the effect of wording from the effect of
   ensembling, or see whether they interact.

Adding **v5 = raw names × ensemble** completes the factorial. Every combination
is then run on both models, on the same test chips, with everything else fixed.

In [ ]:
def prompts_v5(class_names=C.CLASS_NAMES):
    """Raw CamelCase labels x the template ensemble — the missing 2x2 cell."""
    return {c: [t.format(c) for t in CL.PROMPT_TEMPLATES] for c in class_names}

STRATEGIES = dict(CL.PROMPT_STRATEGIES)
STRATEGIES["v5_raw_ensemble"] = prompts_v5

sweep = {}
for model_name, (model, _, tokenizer) in models.items():
    feats_test = torch.from_numpy(FEATS[model_name][test_idx])
    feats_test = feats_test / feats_test.norm(dim=-1, keepdim=True)
    for strat_name, builder in STRATEGIES.items():
        W = CL.build_zeroshot_classifier(model, tokenizer, builder(), device=DEVICE)
        _, pred = CL.zeroshot_predict(feats_test, W)
        sweep[(model_name, strat_name)] = {
            "accuracy": E.accuracy(y_test, pred),
            "macro_f1": E.macro_f1(y_test, pred, 10),
            "pred": pred,
        }

print(f"{'strategy':<24}" + "".join(f"{m:>24}" for m in models))
for strat in STRATEGIES:
    row = f"{strat:<24}"
    for m in models:
        r = sweep[(m, strat)]
        row += f"{r['accuracy']:>12.4f}{r['macro_f1']:>12.4f}"
    print(row)
print(f"\n{'':<24}" + "".join(f"{'acc':>12}{'macroF1':>12}" for _ in models))

### The 2×2, read as a factorial

`wording ∈ {raw CamelCase, natural English}` × `prompts ∈ {single template,
10-template ensemble}`. If the two factors were independent, the two rows would
shift by the same amount. They do not have to.

In [ ]:
FACTORIAL = {
    ("raw", "single"):   "v2_simple_template",
    ("raw", "ensemble"): "v5_raw_ensemble",
    ("natural", "single"):   "v3_natural_names",
    ("natural", "ensemble"): "v4_prompt_ensemble",
}
for model_name in models:
    print(f"\n{model_name} — accuracy")
    print(f"{'':<12}{'single':>10}{'ensemble':>10}{'ensembling effect':>20}")
    for wording in ("raw", "natural"):
        a = sweep[(model_name, FACTORIAL[(wording, "single")])]["accuracy"]
        b = sweep[(model_name, FACTORIAL[(wording, "ensemble")])]["accuracy"]
        print(f"{wording:<12}{a:>10.4f}{b:>10.4f}{b - a:>+20.4f}")
    w_single = (sweep[(model_name, FACTORIAL[("natural", "single")])]["accuracy"]
                - sweep[(model_name, FACTORIAL[("raw", "single")])]["accuracy"])
    w_ens = (sweep[(model_name, FACTORIAL[("natural", "ensemble")])]["accuracy"]
             - sweep[(model_name, FACTORIAL[("raw", "ensemble")])]["accuracy"])
    print(f"{'wording effect':<12}{w_single:>+10.4f}{w_ens:>+10.4f}"
          f"{'   interaction: ' + format(w_ens - w_single, '+.4f')}")

best_per_model = {m: max(STRATEGIES, key=lambda s: sweep[(m, s)]["macro_f1"]) for m in models}
print("\nbest strategy per model (macro-F1):")
for m, s in best_per_model.items():
    print(f"  {m:<12} {s:<24} {sweep[(m, s)]['macro_f1']:.4f}")

if len(models) > 1:
    gap_v4 = sweep[("CLIP", "v4_prompt_ensemble")]["macro_f1"] - sweep[("RemoteCLIP", "v4_prompt_ensemble")]["macro_f1"]
    gap_best = sweep[("CLIP", best_per_model["CLIP"])]["macro_f1"] - sweep[("RemoteCLIP", best_per_model["RemoteCLIP"])]["macro_f1"]
    print(f"\nCLIP - RemoteCLIP gap using CLIP's best prompt: {gap_v4:+.4f}")
    print(f"CLIP - RemoteCLIP gap when EACH model uses its own best prompt: {gap_best:+.4f}")
    print("\nIf the gap barely moves, prompt transfer was not the explanation and the")
    print("RemoteCLIP zero-shot deficit is a property of the model on this data.")

## B — Is the failure in the image tower or the text tower?

This is the decisive experiment, and it is cheap.

A CLIP-style zero-shot classifier has two halves: an **image encoder** producing
features, and a **text encoder** producing one prototype per class. Zero-shot
accuracy confounds them — a bad score can mean bad features *or* well-placed
features with badly-placed text prototypes.

Separating them requires only one substitution. Replace the text prototype for
each class with the **mean image feature of k labelled examples** of that class
(a nearest-class-mean classifier in the same embedding space), and re-measure.
Now the image tower is doing all the work and the text tower is out of the loop.

- If RemoteCLIP's image prototypes beat CLIP's while its text prototypes lose,
  the encoder is *better* and the language alignment is *worse*. That is a
  precise, mechanistic claim.
- If RemoteCLIP loses both ways, its features are simply worse on 10 m imagery
  and the NB04 probe results need re-examining.

NB04 already hints at the answer — RemoteCLIP wins as a frozen probe at every
budget — but a linear probe *learns* a decision boundary. Nearest-class-mean
learns nothing beyond an average, so it is a much more direct read of the
geometry, and it is measured in the same units as the zero-shot number.

In [ ]:
from sklearn.preprocessing import normalize

def nearest_class_mean(feats, k, seed):
    """Class prototypes = mean of k labelled TRAIN features; cosine to test."""
    F_all = normalize(feats)
    idx = D.few_shot_indices(y, train_idx, k=k, seed=seed)
    protos = np.stack([F_all[idx[y[idx] == c]].mean(axis=0) for c in range(10)])
    protos = normalize(protos)
    pred = (F_all[test_idx] @ protos.T).argmax(axis=1)
    return E.macro_f1(y_test, pred, 10)

K_PROTO = (1, 5, 20)
proto_results = {}
for model_name in models:
    proto_results[model_name] = {}
    for k in K_PROTO:
        scores = [nearest_class_mean(FEATS[model_name], k, seed=1000 * d + k) for d in range(5)]
        proto_results[model_name][k] = (float(np.mean(scores)), float(np.std(scores)))

print(f"{'model':<14}{'text prototypes':>18}" + "".join(f"{'image k=' + str(k):>18}" for k in K_PROTO))
for model_name in models:
    zs = sweep[(model_name, best_per_model[model_name])]["macro_f1"]
    row = f"{model_name:<14}{zs:>18.4f}"
    for k in K_PROTO:
        m, s = proto_results[model_name][k]
        row += f"{m:>12.4f}±{s:.3f}"
    print(row)

if len(models) > 1:
    print("\nThe dissociation, stated as two numbers:")
    zs_gap = sweep[("RemoteCLIP", best_per_model["RemoteCLIP"])]["macro_f1"] - \
             sweep[("CLIP", best_per_model["CLIP"])]["macro_f1"]
    im_gap = proto_results["RemoteCLIP"][20][0] - proto_results["CLIP"][20][0]
    print(f"  RemoteCLIP - CLIP, TEXT prototypes (zero-shot):        {zs_gap:+.4f}")
    print(f"  RemoteCLIP - CLIP, IMAGE prototypes (20 labels/class): {im_gap:+.4f}")
    print("\nOpposite signs => better features, worse language alignment.")

### Why would the text prototypes be misplaced? Look at their geometry

Two diagnostics on the text embeddings themselves, no images involved:

1. **Prototype separation.** The mean pairwise cosine similarity between the ten
   class text embeddings. If they are all crowded into one direction, the argmax
   is decided by tiny differences and the classifier collapses onto whichever
   class sits closest to the image manifold — which is exactly the "everything
   predicted as Forest" pattern in NB03's RemoteCLIP confusion matrix.

2. **Text–image agreement.** For each class, is that class's text embedding
   actually the nearest text embedding to the class's own image centroid? The
   diagonal hit-rate of that matrix is a direct measure of language alignment,
   and it needs no classifier at all.

In [ ]:
import matplotlib.pyplot as plt

geometry = {}
fig, axes = plt.subplots(1, len(models), figsize=(6.2 * len(models), 5.4))
for ax, (model_name, (model, _, tokenizer)) in zip(np.atleast_1d(axes), models.items()):
    W = CL.build_zeroshot_classifier(model, tokenizer, STRATEGIES[best_per_model[model_name]](),
                                     device=DEVICE).float().cpu().numpy()
    sim = W @ W.T
    off = sim[~np.eye(10, dtype=bool)]

    F_all = normalize(FEATS[model_name])
    centroids = normalize(np.stack([F_all[train_idx][y[train_idx] == c].mean(axis=0) for c in range(10)]))
    cross = centroids @ W.T                      # (image class, text class)
    diagonal_hits = int((cross.argmax(axis=1) == np.arange(10)).sum())

    geometry[model_name] = {
        "mean_offdiag_text_similarity": float(off.mean()),
        "max_offdiag_text_similarity": float(off.max()),
        "text_image_diagonal_hits": diagonal_hits,
    }
    print(f"{model_name}: mean pairwise text-prototype similarity {off.mean():.3f} "
          f"(max {off.max():.3f});  image centroid finds its own text prototype for "
          f"{diagonal_hits}/10 classes")

    im = ax.imshow(cross, cmap="viridis")
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(C.CLASS_NAMES, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(C.CLASS_NAMES, fontsize=7)
    ax.set_xlabel("text prototype"); ax.set_ylabel("image centroid")
    ax.set_title(f"{model_name}: {diagonal_hits}/10 classes aligned")
    ax.grid(False)
    for i in range(10):
        j = int(cross[i].argmax())
        ax.plot(j, i, marker="s", ms=5, mfc="none", mec="red", mew=1.4)
fig.colorbar(im, ax=list(np.atleast_1d(axes)), fraction=0.02, label="cosine similarity")
fig.suptitle("Do image centroids point at the right text prototype? (red = argmax)")
V.save(fig, "07_text_image_alignment")

## C — How much does the random split actually leak?

NB01 flagged that EuroSAT chips are cut from a limited set of Sentinel-2 scenes,
so a random split puts spatially adjacent chips in both train and test. NB06 then
produced a suspicious number: masking B01/B09/B10 — the **atmospheric** bands,
which carry almost no surface information — costs more macro-F1 (0.076) than
masking the NIR bands (0.023). A model leaning on aerosol and water-vapour
channels is not learning land cover; it is recognising *acquisition conditions*.

**The test.** EuroSAT ships no scene identifiers, so a true spatially-blocked
split is impossible. But we can build a proxy: cluster the chips on their
atmospheric-band statistics alone, and treat each cluster as a pseudo-scene.
Chips sharing an atmospheric fingerprint were probably acquired together, and
holding out whole clusters approximates holding out whole scenes.

Then train the identical model under the identical protocol on both splits. The
difference between them is an estimate of how much the headline 0.984 owes to
leakage rather than to generalisation.

**Honest about the proxy:** clustering on atmosphere is not the same as
clustering on geography, and the blocked split is also harder for a second
reason — its train and test sets have genuinely different illumination
statistics, which is a covariate shift on top of the removed leakage. So the
drop is an *upper bound* on the leakage effect, not a clean estimate. That
distinction is stated in the findings rather than glossed.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

ATMO = ["B01", "B09", "B10"]
atmo_idx = [B.band_index(b) for b in ATMO]

t0 = time.time()
rows = []
for s in range(0, y.size, 2048):
    block = np.asarray(X[s:s + 2048], dtype=np.float32)[:, atmo_idx]
    flat = block.reshape(block.shape[0], len(ATMO), -1)
    rows.append(np.concatenate([flat.mean(axis=2), flat.std(axis=2)], axis=1))
atmo_features = np.concatenate(rows)
print(f"atmospheric fingerprint {atmo_features.shape} in {time.time() - t0:.0f}s "
      f"(mean and std of {ATMO})")

N_CLUSTERS = 25
clusters = KMeans(n_clusters=N_CLUSTERS, random_state=cfg.seed, n_init=10).fit_predict(
    StandardScaler().fit_transform(atmo_features)
)
sizes = np.bincount(clusters)
print(f"{N_CLUSTERS} pseudo-scene clusters, sizes {sizes.min()}–{sizes.max()} "
      f"(median {int(np.median(sizes))})")

**Do these clusters carry class information on their own?** If a pseudo-scene
predicts the label better than chance, then knowing which scene a chip came from
tells you what is in it — which is precisely the channel through which a random
split leaks.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Can the label be predicted from the atmospheric fingerprint alone?
scaler = StandardScaler().fit(atmo_features[train_idx])
clf = LogisticRegression(max_iter=2000, n_jobs=-1).fit(scaler.transform(atmo_features[train_idx]), y[train_idx])
atmo_only_acc = E.accuracy(y_test, clf.predict(scaler.transform(atmo_features[test_idx])))
atmo_only_f1 = E.macro_f1(y_test, clf.predict(scaler.transform(atmo_features[test_idx])), 10)
print(f"class predicted from 6 atmospheric-band statistics alone: "
      f"acc {atmo_only_acc:.4f}, macro-F1 {atmo_only_f1:.4f}  (chance = 0.100)")

# How concentrated is each class within a few clusters?
contingency = np.zeros((N_CLUSTERS, 10))
for c, lab in zip(clusters, y):
    contingency[c, lab] += 1
purity = (contingency.max(axis=1) / np.maximum(contingency.sum(axis=1), 1)).mean()
print(f"mean cluster purity (fraction of a cluster that is its dominant class): {purity:.3f} "
      f"(a perfectly class-agnostic clustering would give ~{1/10:.3f}–0.20)")
print("\nCaveat before over-reading this: land cover genuinely affects B01 through")
print("surface brightness and aerosol scattering, so some of this signal is physical")
print("rather than leakage. The blocked-split experiment below is the stronger test.")

### Build the blocked split and retrain

Clusters are assigned whole to train/val/test, greedily, so that the split sizes
land near 70/15/15. The per-class proportions are then printed: unlike the
stratified random split, a blocked split **cannot** guarantee class balance, and
that imbalance is part of what makes it harder — it is reported rather than
engineered away.

Both models are trained here, in this notebook, under identical settings — 15
epochs, one seed — rather than comparing against NB02's numbers, so the
comparison is not contaminated by a difference in epochs or protocol.

In [ ]:
order = np.argsort(sizes)[::-1]
targets = {"train": 0.70, "val": 0.15, "test": 0.15}
assigned = {k: [] for k in targets}
totals = {k: 0 for k in targets}
for cluster_id in order:
    deficits = {k: targets[k] - totals[k] / max(sum(totals.values()), 1) for k in targets}
    pick = max(deficits, key=deficits.get)
    assigned[pick].append(cluster_id)
    totals[pick] += sizes[cluster_id]

blocked = {k: np.sort(np.flatnonzero(np.isin(clusters, v))) for k, v in assigned.items()}
for k, v in blocked.items():
    print(f"{k:<6}{v.size:>7,} chips ({v.size / y.size:.1%})  from {len(assigned[k])} clusters")

overall = np.bincount(y, minlength=10) / y.size
print(f"\n{'class':<24}{'overall':>9}{'blk train':>11}{'blk test':>10}")
for c, name in enumerate(C.CLASS_NAMES):
    tr = (y[blocked["train"]] == c).mean()
    te = (y[blocked["test"]] == c).mean()
    print(f"{name:<24}{overall[c]:>9.3f}{tr:>11.3f}{te:>10.3f}")
assert min(np.bincount(y[blocked["train"]], minlength=10)) > 0, "a class is absent from blocked train"

Train the same architecture twice, changing only which chips are in which split.
15 epochs and one seed each — enough to size the effect, not enough to claim
three decimal places, which is why the findings below report it as a bound.

In [ ]:
from s2map import train as T, transforms as TR

VERIFY_EPOCHS = 15
verify_cfg = C.TrainConfig(epochs=VERIFY_EPOCHS, batch_size=cfg.train.batch_size, lr=cfg.train.lr,
                           patience=8, num_workers=cfg.train.num_workers, amp=cfg.train.amp)

def loaders_for(split_dict, seed=0):
    ds = {k: D.EuroSATChips(X, y, v, stats, TR.train_transform() if k == "train" else None)
          for k, v in split_dict.items()}
    return (D.make_loader(ds["train"], verify_cfg.batch_size, True, verify_cfg.num_workers, seed),
            D.make_loader(ds["val"], 256, False, verify_cfg.num_workers, seed),
            D.make_loader(ds["test"], 256, False, verify_cfg.num_workers, seed))

split_scores = {}
for label, split_dict in (("random (as used throughout)", splits), ("blocked by pseudo-scene", blocked)):
    C.set_seed(0)
    tr, va, te = loaders_for(split_dict)
    model, hist = T.fit(M.build_resnet18(13, 10, pretrained=True), tr, va, verify_cfg, DEVICE, verbose=False)
    logits, labels = T.predict(model, te, DEVICE)
    split_scores[label] = {
        "accuracy": E.accuracy(labels, logits.argmax(1)),
        "macro_f1": E.macro_f1(labels, logits.argmax(1), 10),
        "val_macro_f1": hist.best_val_macro_f1,
        "epochs": hist.epochs_run,
    }
    print(f"{label:<30} test acc {split_scores[label]['accuracy']:.4f}  "
          f"macro-F1 {split_scores[label]['macro_f1']:.4f}  ({hist.epochs_run} epochs)")

drop = split_scores["random (as used throughout)"]["macro_f1"] - split_scores["blocked by pseudo-scene"]["macro_f1"]
print(f"\nmacro-F1 drop when whole pseudo-scenes are held out: {drop:+.4f}")
print("This is an UPPER BOUND on the leakage effect: the blocked split also imposes")
print("an illumination covariate shift and an unbalanced class distribution.")

## D — Error bars for the two NB02 ablations *(optional, ~25 minutes)*

NB02 ran the RGB-only and random-stem ablations at a single seed, and the two
deltas it reported (+0.0053 for multispectral, +0.0020 for the pretrained stem)
straddle the ±0.0023 seed noise of the main arm. One of them is therefore not
evidence of anything, and neither can be defended without error bars.

This section runs both at three seeds with NB02's exact settings — 20 epochs, so
the numbers remain comparable with that chapter. Set `RUN_SECTION_D = False` to
skip it if the session is short.

In [ ]:
RUN_SECTION_D = True

ablation_results = {}
if RUN_SECTION_D:
    ab_cfg = C.TrainConfig(epochs=20, batch_size=cfg.train.batch_size, lr=cfg.train.lr,
                           patience=10, num_workers=cfg.train.num_workers, amp=cfg.train.amp)

    def loaders_seeded(seed, band_subset=None):
        ds = {k: D.EuroSATChips(X, y, v, stats, TR.train_transform() if k == "train" else None, band_subset)
              for k, v in splits.items()}
        return (D.make_loader(ds["train"], ab_cfg.batch_size, True, ab_cfg.num_workers, seed),
                D.make_loader(ds["val"], 256, False, ab_cfg.num_workers, seed),
                D.make_loader(ds["test"], 256, False, ab_cfg.num_workers, seed))

    arms = {
        "resnet18_rgb": (lambda s: M.build_resnet18(3, 10, pretrained=True),
                         lambda s: loaders_seeded(s, list(B.RGB_INDICES))),
        "resnet18_randomstem": (lambda s: M.build_resnet18(13, 10, pretrained=True, stem_mode="random"),
                                lambda s: loaders_seeded(s)),
    }
    for arm_name, (model_fn, loader_fn) in arms.items():
        res = T.fit_multi_seed(model_fn, loader_fn, seeds=cfg.seeds, train_cfg=ab_cfg,
                               device=DEVICE, verbose=False)
        ablation_results[arm_name] = res
        print(f"{arm_name:<24} macro-F1 {res['test_macro_f1']['mean']:.4f} ± "
              f"{res['test_macro_f1']['std']:.4f}  (seeds {list(cfg.seeds)})")
else:
    print("section D skipped")

Compare each ablation against the 3-seed main arm from NB02, and judge the delta
against the *pooled* seed noise of the two rather than against zero.

In [ ]:
if ablation_results:
    base = prior_value("02", "arm2_resnet18_13band")
    base_std = next(r["test_macro_f1"]["std"] for r in prior
                    if r.get("notebook") == "02" and r.get("arm") == "arm2_resnet18_13band")
    print(f"13-band inflated stem (NB02, 3 seeds):  {base:.4f} ± {base_std:.4f}\n")
    for arm_name, res in ablation_results.items():
        m, s = res["test_macro_f1"]["mean"], res["test_macro_f1"]["std"]
        delta = base - m
        pooled = float(np.sqrt(base_std ** 2 + s ** 2))
        verdict = "SUPPORTED" if abs(delta) > 2 * pooled else "NOT SUPPORTED (within noise)"
        print(f"{arm_name:<24} {m:.4f} ± {s:.4f}   delta {delta:+.4f}  "
              f"pooled sigma {pooled:.4f}  -> {verdict}")
    print("\n'delta > 2 pooled sigma' is a deliberately crude bar. With 3 seeds a real")
    print("significance test is not available; the point is to avoid claiming an effect")
    print("that is smaller than the noise we can already see.")

### Record everything for the write-up

In [ ]:
entry = {
    "notebook": "07", "arm": "verification",
    "prompt_sweep": {f"{m}|{s}": {"accuracy": r["accuracy"], "macro_f1": r["macro_f1"]}
                     for (m, s), r in sweep.items()},
    "best_strategy_per_model": best_per_model,
    "image_prototype_macro_f1": {m: {str(k): v for k, v in d.items()} for m, d in proto_results.items()},
    "text_embedding_geometry": geometry,
    "atmospheric_only_accuracy": atmo_only_acc,
    "atmospheric_only_macro_f1": atmo_only_f1,
    "cluster_purity": float(purity),
    "split_comparison": split_scores,
    "leakage_upper_bound_macro_f1_drop": float(drop),
    "ablations_3_seed": {k: {"mean": v["test_macro_f1"]["mean"], "std": v["test_macro_f1"]["std"]}
                         for k, v in ablation_results.items()},
    "notes": "confirmatory analyses of NB03/NB06 results; blocked split is an atmospheric-fingerprint proxy",
}
E.append_result(entry)
print("results.json updated\n")

print("=" * 72)
print("SUMMARY FOR THE WRITE-UP")
print("=" * 72)
if len(models) > 1:
    print(f"1. RemoteCLIP zero-shot deficit with its OWN best prompt: {gap_best:+.4f} macro-F1")
    print(f"   -> prompt transfer {'WAS NOT' if abs(gap_best) > 0.05 else 'MAY HAVE BEEN'} the explanation")
    print(f"2. Image prototypes (20/class): RemoteCLIP - CLIP = {im_gap:+.4f}")
    print(f"   Text prototypes (zero-shot): RemoteCLIP - CLIP = {zs_gap:+.4f}")
    print(f"   -> {'DISSOCIATION CONFIRMED' if im_gap > 0 > zs_gap else 'no dissociation'}")
    for m in models:
        g = geometry[m]
        print(f"   {m}: text prototypes align with own image centroid for "
              f"{g['text_image_diagonal_hits']}/10 classes")
print(f"3. Leakage upper bound: {drop:+.4f} macro-F1 when pseudo-scenes are held out")
print(f"   class predictable from atmospheric bands alone at {atmo_only_acc:.1%} (chance 10%)")
if ablation_results:
    print("4. Ablation error bars:")
    for arm_name, res in ablation_results.items():
        print(f"   {arm_name}: {res['test_macro_f1']['mean']:.4f} ± {res['test_macro_f1']['std']:.4f}")

## Findings

_Fill in from the run — but the shape of each conclusion is determined by which
way the numbers came out, and all four outcomes are worth reporting._

1. **The RemoteCLIP deficit is / is not a prompting artefact.** Section A gives
   each model its own best prompt. If the gap survives, the zero-shot result
   stands as a property of the model on 10 m imagery.

2. **Better features, worse language alignment.** Section B separates the two
   towers. Opposite signs — RemoteCLIP ahead on image prototypes, behind on text
   prototypes — is the mechanistic explanation for a result that looks like a
   contradiction: NB03 says RemoteCLIP is worse, NB04 says it is better, and both
   are right about different halves of the model.

3. **The text-prototype geometry shows why.** Crowded class prototypes and a low
   image-centroid-to-text diagonal hit-rate mean the argmax is decided by noise,
   which is what produces a confusion matrix that collapses onto one class.

4. **The random split leaks, and here is the upper bound.** Holding out whole
   atmospheric pseudo-scenes costs the measured drop. Reported as an upper bound
   because the blocked split adds a covariate shift and class imbalance on top of
   removing the leakage. Together with NB06's band ablation — atmospheric bands
   mattering more than NIR — this turns a boilerplate limitation into a measured
   one.

5. **The NB02 ablations now have error bars**, and at least one of the two deltas
   is expected to fall inside the noise. Reporting an effect smaller than the
   seed spread as though it were real is exactly the error this section exists to
   prevent.